In [ ]:
from pathlib import Path
import base64
import subprocess

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

REPO_URL = "https://github.com/xl4357/MSSP6070.git"
REPO_DIR = Path("/content/MSSP6070")
CSV_PATH = REPO_DIR / "Assignments" / "InternetSales.csv"

if not CSV_PATH.exists():

    result = subprocess.run(
        ["git", "clone", "--quiet", REPO_URL, str(REPO_DIR)],
        capture_output=True, text=True
    )
    if result.returncode != 0:

        from google.colab import userdata
        token = userdata.get("Git_Key")
        if not token:
            raise RuntimeError("Clone failed. Check repository access and Git_Key secret.")
        auth = base64.b64encode(f"x-access-token:{token}".encode()).decode()
        subprocess.run(
            ["git", "-c", f"http.extraheader=AUTHORIZATION: basic {auth}",
             "clone", "--quiet", REPO_URL, str(REPO_DIR)],
            check=True, capture_output=True
        )

if not CSV_PATH.is_file():
    raise FileNotFoundError(f"Dataset not found: {CSV_PATH}")

sales = pd.read_csv(
    CSV_PATH,
    parse_dates=["BirthDate", "DateFirstPurchase", "OrderDate", "DueDate", "ShipDate"],
    encoding="latin1", low_memory=False
)

required = {
    "CustomerKey", "OrderDate", "EnglishCountryRegionName", "SalesAmount",
    "YearlyIncome", "Gender", "HouseOwnerFlag", "EnglishProductName",
    "ProductKey", "UnitPrice", "SalesOrderNumber"
}
missing = required.difference(sales.columns)
if missing:
    raise ValueError(f"Missing required columns: {sorted(missing)}")

sales["OrderYear"] = sales["OrderDate"].dt.year
print("Rows and columns:", sales.shape)
print("Years:", sorted(sales["OrderYear"].dropna().unique().tolist()))
print("Missing values in analysis columns:")
print(sales[list(sorted(required))].isna().sum().loc[lambda x: x > 0])
assert sales["SalesAmount"].notna().all(), "Review missing sales amounts before aggregation."
assert sales["OrderDate"].notna().all(), "Review missing order dates before annual comparisons."

In [ ]:
year_country_sales = (
    sales.groupby(["OrderYear", "EnglishCountryRegionName"])["SalesAmount"]
    .sum()
    .unstack(fill_value=0)
    .sort_index()
)

sales_millions = year_country_sales / 1_000_000
country_totals_millions = sales_millions.sum().sort_values(ascending=False)

display(sales_millions.round(3))
print("Total sales by country/region (USD millions):")
display(country_totals_millions.round(3).to_frame("Sales_USD_Millions"))

In [ ]:
ax = sales_millions.plot(marker="o", figsize=(11, 5))
ax.set(title="Internet Sales by Year and Country/Region",
       xlabel="Order Year", ylabel="Sales (USD millions)")
ax.set_xticks(sales_millions.index)
ax.grid(alpha=0.25)
ax.legend(title="Country/Region", bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout()
plt.show()

In [ ]:
date_coverage = sales.groupby("OrderYear")["OrderDate"].agg(["min", "max", "count"])
display(date_coverage)

COMPARABLE_YEARS = [2011, 2012, 2013]
if not set(COMPARABLE_YEARS).issubset(year_country_sales.index):
    raise ValueError("At least one comparison year is absent from the data.")

growth_pct = year_country_sales.loc[COMPARABLE_YEARS].pct_change() * 100

growth_pct = growth_pct.iloc[1:]
display(growth_pct.round(2))

fastest_growth = pd.DataFrame({
    "FastestGrowingRegion": growth_pct.idxmax(axis=1),
    "GrowthPercent": growth_pct.max(axis=1)
})
display(fastest_growth.round(2))

In [ ]:
ax = growth_pct.T.plot.bar(figsize=(11, 5))
ax.axhline(0, color="black", linewidth=0.8)
ax.set(title="Year-over-Year Sales Growth by Country/Region",
       xlabel="Country/Region", ylabel="Growth (%)")
plt.xticks(rotation=35, ha="right")
plt.tight_layout()
plt.show()

In [ ]:
sales_change_millions = (
    year_country_sales.loc[COMPARABLE_YEARS].diff().iloc[1:] / 1_000_000
)
print("Year-over-year change in sales (USD millions):")
display(sales_change_millions.round(3))

In [ ]:
profile_cols = ["EnglishCountryRegionName", "YearlyIncome", "Gender", "HouseOwnerFlag"]
profile_variation = sales.groupby("CustomerKey")[profile_cols].nunique(dropna=False)
conflicts = profile_variation.gt(1).any(axis=1)
print("Customers with conflicting profile values:", int(conflicts.sum()))
if conflicts.any():
    display(profile_variation.loc[conflicts].head(10))
    raise ValueError("Conflicting customer profiles detected. Resolve before customer-level analysis.")

customers = sales[["CustomerKey", *profile_cols]].drop_duplicates(subset="CustomerKey").copy()
assert customers["CustomerKey"].is_unique
print("Unique customers:", len(customers))

In [ ]:
income_summary = (
    customers.groupby("EnglishCountryRegionName")["YearlyIncome"]
    .agg(Customers="count", Minimum="min", Maximum="max",
         Mean="mean", StandardDeviation="std")
)
display(income_summary.round(2))

gender_counts = pd.crosstab(customers["EnglishCountryRegionName"], customers["Gender"])
gender_percent = gender_counts.div(gender_counts.sum(axis=1), axis=0) * 100
print("Gender counts:")
display(gender_counts)
print("Gender distribution (% within country/region):")
display(gender_percent.round(2))

In [ ]:
ax = gender_percent.plot.bar(stacked=True, figsize=(10, 5))
ax.set(title="Gender Distribution by Country/Region",
       xlabel="Country/Region", ylabel="Customers (%)")
ax.legend(title="Gender", bbox_to_anchor=(1.02, 1), loc="upper left")
plt.xticks(rotation=35, ha="right")
plt.tight_layout()
plt.show()

In [ ]:
def classify_product(name):
    if pd.isna(name):
        return "Unclassified"
    name = str(name)
    if name.startswith(("Mountain-", "Road-", "Touring-")):
        return "Bikes"
    if any(term in name for term in ("Jersey", "Socks", "Gloves", "Vest", "Shorts")):
        return "Clothing"
    if any(term in name for term in ("Tire", "Tube")):
        return "Components"

    if any(term in name for term in (
        "Helmet", "Bottle", "Cage", "Pump", "Lock", "Fender", "Rack",
        "Clean", "Patch", "Hydration", "Cap", "Stand"
    )):
        return "Accessories"
    return "Unclassified"

sales["ProductCategory"] = sales["EnglishProductName"].map(classify_product)
unknown_products = (
    sales.loc[sales["ProductCategory"].eq("Unclassified"),
              ["ProductKey", "EnglishProductName"]]
    .drop_duplicates().sort_values("EnglishProductName")
)
print("Unclassified distinct products:", len(unknown_products))
display(unknown_products.head(50))

In [ ]:
category_summary = (
    sales.groupby("ProductCategory")
    .agg(TotalSales=("SalesAmount", "sum"),
         DistinctProducts=("ProductKey", "nunique"),
         SalesLines=("SalesAmount", "size"))
    .sort_values("TotalSales", ascending=False)
)
category_summary["SalesSharePercent"] = (
    100 * category_summary["TotalSales"] / category_summary["TotalSales"].sum()
)
display(category_summary.round(2))

product_price_check = sales.groupby("ProductKey")["UnitPrice"].nunique(dropna=False)
if product_price_check.gt(1).any():
    print("Caution: Some ProductKeys have multiple recorded UnitPrice values.")
    print("The distinct-product mean below uses the first observed price per product.")

products = sales.drop_duplicates("ProductKey")[["ProductKey", "ProductCategory", "UnitPrice"]]
mean_listed_price = products.groupby("ProductCategory")["UnitPrice"].mean()

mean_line_price = sales.groupby("ProductCategory")["UnitPrice"].mean()
price_comparison = pd.DataFrame({
    "MeanPricePerDistinctProduct": mean_listed_price,
    "MeanPricePerSalesLine": mean_line_price
})
display(price_comparison.round(2))

winner = category_summary.index[0]
print(f"Top category: {winner}")
print(f"Average distinct-product price: ${mean_listed_price.loc[winner]:,.2f}")

In [ ]:
ax = (category_summary["TotalSales"] / 1_000_000).plot.bar(figsize=(9, 5))
ax.set(title="Internet Sales by Inferred Product Category",
       xlabel="Category", ylabel="Sales (USD millions)")
plt.xticks(rotation=25, ha="right")
plt.tight_layout()
plt.show()

In [ ]:
customer_sales = (
    sales.groupby("CustomerKey")
    .agg(TotalSales=("SalesAmount", "sum"),
         Orders=("SalesOrderNumber", "nunique"))
    .join(customers.set_index("CustomerKey")[["HouseOwnerFlag"]])
    .reset_index()
)

home_summary = (
    customer_sales.groupby("HouseOwnerFlag")
    .agg(Customers=("CustomerKey", "nunique"),
         AverageCustomerSales=("TotalSales", "mean"),
         MedianCustomerSales=("TotalSales", "median"),
         TotalSales=("TotalSales", "sum"),
         AverageOrders=("Orders", "mean"))
)
home_summary["CustomerSharePercent"] = 100 * home_summary["Customers"] / home_summary["Customers"].sum()
home_summary["SalesSharePercent"] = 100 * home_summary["TotalSales"] / home_summary["TotalSales"].sum()
display(home_summary.round(2))

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
labels = {0: "Non-homeowners", 1: "Homeowners"}
values = home_summary["AverageCustomerSales"]
ax.bar([labels.get(k, str(k)) for k in values.index], values.to_numpy())
ax.set(title="Average Cumulative Sales per Customer by Homeownership",
       ylabel="Sales per Customer (USD)")
plt.tight_layout()
plt.show()

In [ ]:
homeowners = customer_sales.loc[customer_sales["HouseOwnerFlag"].eq(1), "TotalSales"].dropna()
nonhomeowners = customer_sales.loc[customer_sales["HouseOwnerFlag"].eq(0), "TotalSales"].dropna()
if min(len(homeowners), len(nonhomeowners)) < 2:
    raise ValueError("Both groups need at least two customers for Welch's t-test.")

result = stats.ttest_ind(homeowners, nonhomeowners, equal_var=False)
difference = homeowners.mean() - nonhomeowners.mean()
relative_difference_pct = 100 * difference / nonhomeowners.mean()

a = homeowners.var(ddof=1) / len(homeowners)
b = nonhomeowners.var(ddof=1) / len(nonhomeowners)
se = np.sqrt(a + b)
df = (a + b)**2 / (a**2 / (len(homeowners) - 1) + b**2 / (len(nonhomeowners) - 1))
ci_low, ci_high = stats.t.interval(0.95, df, loc=difference, scale=se)

print(f"Group sizes: homeowners={len(homeowners)}, non-homeowners={len(nonhomeowners)}")
print(f"Difference in mean cumulative customer sales: ${difference:,.2f}")
print(f"Relative difference: {relative_difference_pct:.2f}%")
print(f"Welch t-statistic: {result.statistic:.4f}")
print(f"Two-sided p-value: {result.pvalue:.6g}")
print(f"95% confidence interval for mean difference: (${ci_low:,.2f}, ${ci_high:,.2f})")

In [ ]:
raw_total = sales["SalesAmount"].sum()
assert np.isclose(year_country_sales.to_numpy().sum(), raw_total)
assert np.isclose(category_summary["TotalSales"].sum(), raw_total)
assert np.isclose(customer_sales["TotalSales"].sum(), raw_total)
assert len(customer_sales) == customers["CustomerKey"].nunique()

print("Validation passed: annual, category, and customer totals reconcile.")
print("Total revenue (USD):", round(raw_total, 2))
print("Unique customers:", customers["CustomerKey"].nunique())
print("Unclassified distinct products:", len(unknown_products))